# Sweeping a single angle


This notebook walks us through a single-angle solve of the transport operator in several geometries.

In other words, it calculates for a fixed angle $\mathbf{\Omega}_k$ and energy group $g$:

$$\mathbf{T}^{-1}_{gk} [q_{gkei}] $$



In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu" 
import jax
import jax.numpy as jnp
import numpy as onp

We are first going to create a simple slab domain in 1D:

In [ ]:
n_vertices = 20

vertices     = jnp.linspace(0, 1, n_vertices)[:, None]
connectivity = jnp.array([[i, i + 1] for i in range(n_vertices - 1)])
mesh         = (vertices, connectivity)

from jax_sn.domain import CrossSectionData, CrossSectionsIndexed, Domain
from jax_sn.solution_domain import BasixLagrangianSimplex

cross_section = CrossSectionData(
    total     = jnp.array([[1.0]]), 
    scattering= jnp.array([[[[0.0]]]]))

cross_sections_indexed = CrossSectionsIndexed(indices = jnp.full((n_vertices - 1,), 0), cross_section_data = cross_section)

domain = Domain.from_mesh_and_cross_sections(mesh, BasixLagrangianSimplex(1,1).face_template, cross_sections_indexed) # face template is [[0],[1]] first face is first node, second is second.

Let's pick a particular angle to invert the transport operator

In [ ]:
angle = jnp.array([-1.0, 0.0, 0.0]) 

Technically, a transport operator may require different data: a level-based sweep is different then a topologically ordered one. 
Here just create the basic variant:


In [ ]:
from jax_sn.operators.transport_operator import BasicSweepData

sweep_data = BasicSweepData.from_domain_angle(domain, angle)

print("Data structure         ", sweep_data)
print("Levels of each element:", sweep_data.levels)
print("Topological sort:      ", sweep_data.levels_argsort)

Unsurprisingly, with a negative angle, the level of the left-most element is 18 (the highest) while the topological sort determines that 
the first element to be solved is 18 (i.e. the right-most-element).

We can also visualize this (below we will do the same for 2D which is actually interesting):

In [ ]:
from jax_sn.tools.plotly import plot_domain
plot_domain(domain, show_graph=True, graph_angle = jnp.array([1.0,0.0,0.0]), show_levels=True)